# HW 2

Aloysius Rebeiro

PUID: 0033534371

In [2]:
import cv2
import numpy as np
import glob
import os
import xml.etree.ElementTree as ET
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import math

## Q1 code

In [6]:
ID_TO_NAME = {
    '0': 'Retractor', 
    '1': 'Clipper', 
    '2': 'Hemostat', 
    '3': 'Scissors', 
    '4': 'Scalpel', 
    '5': 'Hook', 
    '6': 'Forceps'
}


# Define the instrument categories provided
INSTRUMENTS = ['Retractor', 'Clipper', 'Hemostat', 'Scissors', 'Scalpel', 'Hook', 'Forceps']

def extract_features(image, method='ORB'):
    """
    Extracts keypoints and descriptors using either ORB or BRIEF.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    if method == 'ORB':
        detector = cv2.ORB_create()
        keypoints, descriptors = detector.detectAndCompute(gray, None)
    elif method == 'BRIEF':
        # BRIEF requires a separate detector like FAST to find keypoints first
        star = cv2.xfeatures2d.StarDetector_create()
        brief = cv2.xfeatures2d.BriefDescriptorExtractor_create()
        keypoints = star.detect(gray, None)
        keypoints, descriptors = brief.compute(gray, keypoints)
    else:
        raise ValueError("Method must be 'ORB' or 'BRIEF'")
        
    return keypoints, descriptors

def get_axes_and_centroid(contour):
    """
    Calculates the centroid, major axis, and minor axis.
    """
    M = cv2.moments(contour)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = 0, 0

    rect = cv2.minAreaRect(contour)
    return (cX, cY), rect

def parse_xml_annotation(xml_path):
    """
    Parses your specific XML format with rotated bounding boxes (<robndbox>) 
    and integer class names.
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    objects = []
    
    for obj in root.findall('object'):
        # Get the integer name and map it to the string
        name_id = obj.find('name').text
        name = ID_TO_NAME.get(name_id, 'Unknown')
        
        # Parse the rotated bounding box parameters
        robndbox = obj.find('robndbox')
        if robndbox is not None:
            cx = float(robndbox.find('cx').text)
            cy = float(robndbox.find('cy').text)
            w = float(robndbox.find('w').text)
            h = float(robndbox.find('h').text)
            angle_radians = float(robndbox.find('angle').text)
            
            objects.append({
                'name': name, 
                'rbbox': (cx, cy, w, h, angle_radians)
            })
    return objects

def get_upright_bbox_from_rbbox(rbbox, img_shape):
    """
    Calculates a standard axis-aligned bounding box from the rotated parameters 
    so we can easily crop the Region of Interest (ROI).
    """
    cx, cy, w, h, angle_radians = rbbox
    # Convert radians to degrees for OpenCV
    angle_degrees = math.degrees(angle_radians)
    
    # Create the rotated rectangle structure and get its 4 corners
    rect = ((cx, cy), (w, h), angle_degrees)
    box = cv2.boxPoints(rect)
    box = np.int0(box)
    
    # Calculate the upright bounding box that encloses the rotated one
    x, y, w_bound, h_bound = cv2.boundingRect(box)
    
    # Ensure the bounds don't go outside the image dimensions
    y_start = max(0, y)
    x_start = max(0, x)
    y_end = min(img_shape[0], y + h_bound)
    x_end = min(img_shape[1], x + w_bound)
    
    return x_start, y_start, x_end, y_end

def build_database_from_split(train_image_paths, method='ORB'):
    """
    Extracts descriptors from the training split using the updated XML parser.
    """
    database_descriptors = {name: [] for name in ID_TO_NAME.values()}
    
    for img_path in train_image_paths:
        img = cv2.imread(img_path)
        if img is None: continue
            
        # Match your screenshot's folder structure for the XMLs
        xml_path = img_path.replace('.JPG', '.xml').replace('training set/training_set', 'Annotation_set/training_set')
        
        import os
        if not os.path.exists(xml_path):
            continue
            
        annotations = parse_xml_annotation(xml_path)
        
        for ann in annotations:
            name = ann['name']
            
            # Crop the instrument using the new bounding box logic
            x1, y1, x2, y2 = get_upright_bbox_from_rbbox(ann['rbbox'], img.shape)
            roi = img[y1:y2, x1:x2]
            
            if roi.size == 0: continue
                
            _, desc = extract_features(roi, method)
            if desc is not None:
                database_descriptors[name].append(desc)
                
    for name in database_descriptors:
        if len(database_descriptors[name]) > 0:
            database_descriptors[name] = np.vstack(database_descriptors[name])
        else:
            database_descriptors[name] = None
            
    return database_descriptors

def find_instruments(img, database_descriptors, method='ORB'):
    """
    Processes an image, detects instruments, and predicts their labels.
    Returns a list of predicted labels and their centroids.
    """
    output_img = img.copy()
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 100, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    bf = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=True)
    predictions = []
    
    for cnt in contours:
        if cv2.contourArea(cnt) < 500:
            continue
            
        x, y, w, h = cv2.boundingRect(cnt)
        centroid, rect = get_axes_and_centroid(cnt)
        
        # Extract ROI and Features
        roi = img[y:y+h, x:x+w]
        _, roi_desc = extract_features(roi, method=method)
        
        # Recognize Instrument
        best_match_name = "Unknown"
        max_matches = 0
        
        if roi_desc is not None:
            for inst_name, train_desc in database_descriptors.items():
                if train_desc is not None:
                    matches = bf.match(roi_desc, train_desc)
                    if len(matches) > max_matches:
                        max_matches = len(matches)
                        best_match_name = inst_name
                        
        predictions.append({'predicted_name': best_match_name, 'centroid': centroid})
        
        # Draw annotations
        cv2.rectangle(output_img, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.circle(output_img, centroid, 5, (255, 0, 0), -1)
        box = np.int0(cv2.boxPoints(rect))
        cv2.line(output_img, tuple(box[0]), tuple(box[2]), (255, 0, 0), 2)
        cv2.line(output_img, tuple(box[1]), tuple(box[3]), (255, 0, 0), 2)
        cv2.putText(output_img, best_match_name, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

    return predictions, output_img

def evaluate_k_fold(annotation_folder, method='ORB'):
    """
    Trains and tests using LOOCV over the annotated folder.
    """
    # Assuming images are stored as .JPG based on the provided screenshots
    image_paths = glob.glob(os.path.join(annotation_folder, "*.JPG"))
    k = len(image_paths)
    
    y_true_all = []
    y_pred_all = []
    
    for i in range(k):
        print(f"Processing fold {i+1}/{k} - Testing on {os.path.basename(image_paths[i])}")
        test_image_path = image_paths[i]
        train_image_paths = image_paths[:i] + image_paths[i+1:]
        
        # 1. Train
        database_descriptors = build_database_from_split(train_image_paths, method)
        
        # 2. Test Ground Truth
        xml_path = test_image_path.replace('.JPG', '.xml').replace('.jpg', '.xml')
        ground_truth_objects = parse_xml_annotation(xml_path)
        
        # 3. Predict
        test_img = cv2.imread(test_image_path)
        predictions, output_img = find_instruments(test_img, database_descriptors, method)
        
        # Optional: Save or show the output image for the specific test fold
        # cv2.imwrite(f"output_fold_{i}.jpg", output_img)
        
        # 4. Map predictions to ground truth (simplified distance check)
        for gt in ground_truth_objects:
            gt_name = gt['name']
            gt_xmin, gt_ymin, gt_xmax, gt_ymax = gt['bbox']
            gt_center = ((gt_xmin + gt_xmax) // 2, (gt_ymin + gt_ymax) // 2)
            
            y_true_all.append(gt_name)
            
            # Find closest prediction by centroid distance to match detected obj to GT obj
            closest_pred = "Unknown"
            min_dist = float('inf')
            
            for pred in predictions:
                px, py = pred['centroid']
                dist = np.sqrt((gt_center[0] - px)**2 + (gt_center[1] - py)**2)
                if dist < min_dist and dist < 150: # 150 pixels distance threshold
                    min_dist = dist
                    closest_pred = pred['predicted_name']
                    
            y_pred_all.append(closest_pred)
            
    return y_true_all, y_pred_all

def print_confusion_mat(y_true, y_pred):
    labels = INSTRUMENTS + ["Unknown"]
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    print("\nConfusion Matrix:")
    print(f"{'':<12} " + " ".join([f"{label[:7]:<8}" for label in labels]))
    for i, row_label in enumerate(labels):
        row_str = f"{row_label:<12} " + " ".join([f"{val:<8}" for val in cm[i]])
        print(row_str)

# --- Execution ---
# Set this path to where your Annotation_set/training_set is located locally
# Use an 'r' before the string to handle Windows backslashes correctly
ANNOTATION_FOLDER = r"C:\Users\Aloysius Brave - purdue.edu\Desktop\purdue class stuff\IE 549\HW2\Annotation_set\training_set"

# Run the K-fold evaluation
true_labels, pred_labels = evaluate_k_fold(ANNOTATION_FOLDER, method='ORB')
print_confusion_mat(true_labels, pred_labels)


Confusion Matrix:
             Retract  Clipper  Hemosta  Scissor  Scalpel  Hook     Forceps  Unknown 
Retractor    0        0        0        0        0        0        0        0       
Clipper      0        0        0        0        0        0        0        0       
Hemostat     0        0        0        0        0        0        0        0       
Scissors     0        0        0        0        0        0        0        0       
Scalpel      0        0        0        0        0        0        0        0       
Hook         0        0        0        0        0        0        0        0       
Forceps      0        0        0        0        0        0        0        0       
Unknown      0        0        0        0        0        0        0        0       
